# BCI neural decoding: synthetic spike counts to hand velocity

This beginner tutorial builds a **linear Ridge decoder** that estimates 2-D hand velocity from binned neural spike counts. It is inspired by the general neural-decoding direction of brain-computer interface (BCI) research, including the FALCON benchmark effort, but it runs **entirely locally** by default.

> **Plain-language objective:** learn a mapping from patterns of spike counts across neurons to the hand's horizontal and vertical velocity. We will evaluate that mapping on later, held-out time bins.


## Minimal prerequisites

- Basic Python: variables, arrays, and running notebook cells.
- A local Python environment with `numpy` and `matplotlib` installed. For example, in a terminal: `python -m pip install numpy matplotlib`.
- No account, network connection, neural recording, or download is needed for the main tutorial.

## Quick start

1. Start at **Setup** below.
2. Run the required cells **top-to-bottom**, one at a time (or use “Run All”).
3. Leave the **optional DANDI/FALCON orientation** cell off unless you deliberately want to explore data access later.

The dataset is deterministic: rerunning the notebook with the same code produces the same synthetic observations and results.

## 1. Setup

We use only NumPy for the decoder and Matplotlib for one figure. A fixed random seed makes the simulated recording reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
plt.rcParams["figure.dpi"] = 120


## 2. Make a small synthetic recording

A real neural recording contains spike times. A common first preprocessing step counts each neuron's spikes in short, equal-width time bins. Here each row of `spike_counts` is one 50 ms bin and each column is one simulated neuron.

For teaching purposes, the hand velocity is a smooth 2-D signal, and each neuron's firing rate has a different linear relationship to that signal. We then sample noisy integer counts from Poisson distributions. This is a simplified simulation, **not** a realistic full model of motor cortex or a clinical device.

In [ ]:
bin_width_s = 0.050
n_time_bins = 500
n_neurons = 24
time_s = np.arange(n_time_bins) * bin_width_s

# Target: x and y hand velocity in arbitrary units.
hand_velocity = np.column_stack([
    0.8 * np.sin(2 * np.pi * 0.12 * time_s) + 0.25 * np.sin(2 * np.pi * 0.31 * time_s),
    0.7 * np.cos(2 * np.pi * 0.09 * time_s + 0.4),
])

# Each neuron has a baseline rate and a preferred velocity direction.
baseline_hz = rng.uniform(5.0, 18.0, size=n_neurons)
tuning = rng.normal(0.0, 7.0, size=(2, n_neurons))
rates_hz = np.clip(baseline_hz + hand_velocity @ tuning, 1.0, None)
spike_counts = rng.poisson(rates_hz * bin_width_s)

print(f"spike_counts shape: {spike_counts.shape} (time bins, neurons)")
print(f"hand_velocity shape: {hand_velocity.shape} (time bins, x/y)")


## 3. Create a time-respecting train/test split

We train on the first 70% of time bins and test on the later 30%. This chronological split is deliberate: randomly mixing nearby bins can make an offline score overly optimistic because adjacent neural and movement samples are correlated.

We will also compute feature means and scales using **training data only**. Fitting preprocessing on all bins would let information from the test period influence the model: a form of time leakage.

In [ ]:
train_end = int(0.70 * n_time_bins)
X_train, X_test = spike_counts[:train_end], spike_counts[train_end:]
y_train, y_test = hand_velocity[:train_end], hand_velocity[train_end:]
test_time_s = time_s[train_end:]

# Fit standardization parameters only on the training period.
x_mean = X_train.mean(axis=0)
x_scale = X_train.std(axis=0)
x_scale[x_scale == 0] = 1.0  # Safe for a neuron with constant training counts.
X_train_z = (X_train - x_mean) / x_scale
X_test_z = (X_test - x_mean) / x_scale

print(f"Training bins: {len(X_train)}; test bins: {len(X_test)}")


## 4. Train a Ridge linear decoder

A linear decoder gives every neuron a weight for each velocity component, then adds the weighted counts. **Ridge** regression adds a small penalty on large weights. This often makes a decoder less sensitive to noisy or correlated neural features.

The intercept is handled separately and is not penalized. `ridge_strength` is a modeling choice; in a real project, choose it using validation data that occurs before the final test period.

In [ ]:
ridge_strength = 10.0

# Center targets using training data, then solve the Ridge normal equation.
y_mean = y_train.mean(axis=0)
y_train_centered = y_train - y_mean
identity = np.eye(n_neurons)
weights = np.linalg.solve(
    X_train_z.T @ X_train_z + ridge_strength * identity,
    X_train_z.T @ y_train_centered,
)

# Apply the fitted decoder to held-out, later bins only.
y_pred = X_test_z @ weights + y_mean
print("Decoder trained. Prediction shape:", y_pred.shape)


## 5. Quantify held-out decoding accuracy

We report RMSE (root mean squared error; lower is better) and $R^2$ (variance explained; 1 is perfect, 0 is similar to predicting the test-set mean). Metrics are reported separately for x and y velocity.

Important: these are **offline prediction metrics**, not evidence of real-time BCI control performance. Closed-loop BCI performance also depends on latency, feedback, adaptation, task design, and the person using the system.

In [ ]:
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2, axis=0))
residual_sum_squares = np.sum((y_test - y_pred) ** 2, axis=0)
total_sum_squares = np.sum((y_test - y_test.mean(axis=0)) ** 2, axis=0)
r2 = 1.0 - residual_sum_squares / total_sum_squares

for axis, axis_rmse, axis_r2 in zip(["x", "y"], rmse, r2):
    print(f"{axis} velocity — RMSE: {axis_rmse:.3f}, R²: {axis_r2:.3f}")


## 6. Plot true and decoded velocity

The plot compares the held-out target trajectory with predictions. Look for whether major rises, falls, and direction changes are tracked; do not judge a decoder from a single number alone.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axis_names = ["x velocity", "y velocity"]
for axis_index, ax in enumerate(axes):
    ax.plot(test_time_s, y_test[:, axis_index], label="true", linewidth=2)
    ax.plot(test_time_s, y_pred[:, axis_index], label="Ridge prediction", linewidth=1.5)
    ax.set_ylabel(axis_names[axis_index])
    ax.legend(frameon=False, loc="upper right")
axes[-1].set_xlabel("time (seconds; held-out test period)")
fig.suptitle("Offline decoding on later, held-out time bins")
fig.tight_layout()
plt.show()


## 7. Optional conceptual extension: Kalman/state-space decoding

The Ridge decoder treats each bin independently. A state-space model instead represents an unobserved movement state (for example position and velocity) that evolves smoothly over time, plus neural observations generated from that state. A Kalman filter alternates between:

1. **Predict:** use the previous state and a dynamics model to predict the next state.
2. **Update:** combine that prediction with the new neural observation, accounting for uncertainty.

This can smooth noisy estimates and is common in BCI research, but it introduces modeling assumptions. Its parameters must be trained without test/future information, and evaluation must match the intended causal, real-time use. This notebook does not implement a Kalman filter so the first decoder remains transparent.

## 8. Optional DANDI / FALCON orientation — leave off for the local tutorial

DANDI is a repository for neurophysiology datasets, often in NWB format. FALCON is a neural-decoding benchmark effort with task- and dataset-specific documentation. Moving from this simulation to public data requires checking the current official documentation, the particular dataset's access conditions, signal definitions, timestamps, trial structure, and licenses.

No dandiset identifier is supplied here because it should not be guessed. The opt-in cell below intentionally performs no network operation by default and does not install packages or download data. Set `EXPLORE_DANDI = True` only after you have selected and verified an appropriate dataset from official sources.

In [ ]:
# OPTIONAL: Leave False for the local, synthetic tutorial.
EXPLORE_DANDI = False

if EXPLORE_DANDI:
    print("Orientation only: verify a dataset and its documentation before downloading.")
    print("Then inspect its NWB structure and define a causal train/validation/test protocol.")
else:
    print("DANDI/FALCON exploration is off; no network activity was requested.")


## Takeaways and next steps

You created a reproducible synthetic spike-count dataset, kept later data isolated for testing, trained a Ridge decoder, and checked both metrics and a trajectory plot. Before using real data, define the prediction target and time alignment carefully, reserve genuinely unseen sessions or trials when possible, and ensure every transformation is fitted only on training data.

## References

- DANDI Archive: https://dandiarchive.org/
- DANDI documentation: https://docs.dandiarchive.org/
- FALCON benchmark project: https://falcon-bci.github.io/
- Hochberg, L. R., et al. (2012). Reach and grasp by people with tetraplegia using a neurally controlled robotic arm. *Nature*, 485, 372–375. https://doi.org/10.1038/nature11076
- Wu, W., et al. (2006). Bayesian population decoding of motor cortical activity using a Kalman filter. *Neural Computation*, 18(1), 80–118. https://doi.org/10.1162/089976606774841585

## License

This notebook follows the license of the surrounding `awesome-hands-on-neuroscience` repository. The synthetic data are generated locally when you run the notebook; they are not derived from a participant recording.